# Step 2: Parquet and Partitioning

This notebook converts the raw CSV file into a format a query engine can exploit efficiently: **Parquet**, a columnar, compressed file format with an embedded schema. DuckDB performs the conversion directly, without requiring a separate ETL framework.

The output is also **partitioned** by year and month. A partition is simply a folder: `lake/verkauf/jahr=2026/monat=08/data.parquet` communicates to any engine that everything within it belongs to August 2026, purely through the folder name — no database or catalog is required.

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "requirements.txt").exists():
    os.chdir("..")
print("Working directory:", Path.cwd())

Working directory: /workspaces/python_data_lake


In [2]:
import duckdb

con = duckdb.connect()
con.sql("SELECT * FROM read_csv_auto('lake/raw/sales.csv') LIMIT 5").show()

┌────────────┬────────────┬────────────────┬──────────┬──────────┐
│    date    │   region   │    product     │ quantity │ revenue  │
│    date    │  varchar   │    varchar     │  int64   │  double  │
├────────────┼────────────┼────────────────┼──────────┼──────────┤
│ 2024-03-27 │ Lucerne    │ Power Bank     │       14 │ 16015.28 │
│ 2026-01-18 │ Geneva     │ Webcam HD      │       17 │ 19518.12 │
│ 2025-09-24 │ Zurich     │ Network Switch │       11 │  6928.78 │
│ 2025-02-28 │ Lucerne    │ Router         │        8 │  2616.19 │
│ 2025-02-22 │ Winterthur │ Office Chair   │        1 │   885.63 │
└────────────┴────────────┴────────────────┴──────────┴──────────┘



## Converting to Partitioned Parquet

The statement `COPY ... TO ... (FORMAT PARQUET, PARTITION_BY (...))` reads the CSV, derives `jahr` (year) and `monat` (month) from the date column, and writes one Parquet file per partition folder — all within a single SQL statement.

In [3]:
con.sql("""
    COPY (
        SELECT
            *,
            EXTRACT(year FROM date)  AS jahr,
            EXTRACT(month FROM date) AS monat
        FROM read_csv_auto('lake/raw/sales.csv')
    ) TO 'lake/verkauf' (
        FORMAT PARQUET,
        PARTITION_BY (jahr, monat),
        OVERWRITE_OR_IGNORE 1
    )
""")
print("Done.")

Done.


## Inspecting the Partition Layout

The result is a standard folder tree, viewable in the file explorer or listed directly below.

In [4]:
for path in sorted(Path("lake/verkauf").rglob("*.parquet"))[:12]:
    print(path)

lake/verkauf/jahr=2024/monat=1/data_0.parquet
lake/verkauf/jahr=2024/monat=10/data_0.parquet
lake/verkauf/jahr=2024/monat=11/data_0.parquet
lake/verkauf/jahr=2024/monat=12/data_0.parquet
lake/verkauf/jahr=2024/monat=2/data_0.parquet
lake/verkauf/jahr=2024/monat=3/data_0.parquet
lake/verkauf/jahr=2024/monat=4/data_0.parquet
lake/verkauf/jahr=2024/monat=5/data_0.parquet
lake/verkauf/jahr=2024/monat=6/data_0.parquet
lake/verkauf/jahr=2024/monat=7/data_0.parquet
lake/verkauf/jahr=2024/monat=8/data_0.parquet
lake/verkauf/jahr=2024/monat=9/data_0.parquet


## Comparing File Size: CSV vs. Parquet

A reduction of approximately 5–10x is expected: the columnar layout compresses repeated values (region, product) considerably more effectively than row-based text.

In [5]:
def dir_size_mb(path: str) -> float:
    total_bytes = sum(
        f.stat().st_size for f in Path(path).rglob("*") if f.is_file()
    )
    return total_bytes / (1024 * 1024)

csv_mb = dir_size_mb("lake/raw")
parquet_mb = dir_size_mb("lake/verkauf")

print(f"CSV (raw):          {csv_mb:8.1f} MB")
print(f"Parquet (curated):  {parquet_mb:8.1f} MB")
print(f"Reduction factor:   {csv_mb / parquet_mb:8.1f}x")

CSV (raw):              12.4 MB
Parquet (curated):       2.4 MB
Reduction factor:        5.3x


Two changes occurred, both observable without querying a database:

1. **Compression** — the on-disk footprint was reduced by roughly 5–10x.
2. **Partitioning** — the data is now organized into `jahr=`/`monat=` folders, which the query engine in the next notebook can use to skip irrelevant work entirely.